<a href="https://colab.research.google.com/github/sevenZHQ1018/Econ5200/blob/main/01_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn statsmodels requests -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('All libraries imported successfully!')

All libraries imported successfully!


In [2]:
# Create a production-ready directory structure
dirs = ['data/raw', 'data/processed', 'notebooks', 'figures']
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'Created directory: {d}')

print('\nProject directory structure initialized!')

Created directory: data/raw
Created directory: data/processed
Created directory: notebooks
Created directory: figures

Project directory structure initialized!


In [3]:
# ============================================================
# Download raw data from David Card's Berkeley website
# ============================================================

import urllib.request
import zipfile

DATA_URL = 'https://davidcard.berkeley.edu/data_sets/njmin.zip'

try:
    print('Downloading data...')
    urllib.request.urlretrieve(DATA_URL, 'data/raw/njmin.zip')

    # Unzip the archive
    with zipfile.ZipFile('data/raw/njmin.zip', 'r') as z:
        z.extractall('data/raw/')
        extracted = z.namelist()
        print(f'Extraction complete. Files found: {extracted}')

except Exception as e:
    print(f'Auto-download failed: {e}')
    print('Please manually download: https://davidcard.berkeley.edu/data_sets/njmin.zip')
    print('Then place public.dat in the data/raw/ directory.')

Extraction complete. Files found: ['check.sas', 'codebook', 'public.dat', 'read.me', 'survey1.nj', 'survey2.nj']


In [4]:
# List all files in data/raw/
print('Files in data/raw/:')
for f in os.listdir('data/raw/'):
    size = os.path.getsize(f'data/raw/{f}')
    print(f'  {f:30s}  {size:,} bytes')

Files in data/raw/:
  njmin.zip                       23,017 bytes
  read.me                         1,254 bytes
  codebook                        3,822 bytes
  public.dat                      82,410 bytes
  survey2.nj                      5,480 bytes
  check.sas                       13,770 bytes
  survey1.nj                      5,803 bytes


In [13]:
# ============================================================
# Define column names based on Card & Krueger official codebook
# ============================================================

# Variable descriptions:
# SHEET    : Store ID
# CHAIN    : Chain type (1=Burger King, 2=KFC, 3=Roy Rogers, 4=Wendy's)
# CO_OWNED : Company-owned store dummy (1=yes)
# STATE    : State (1=NJ, 0=PA)
# SOUTHJ   : South Jersey dummy
# CENTRALJ : Central Jersey dummy
# NORTHJ   : North Jersey dummy
# PA1      : Northeast Philadelphia dummy
# PA2      : Easton etc. PA dummy
# SHORE    : Shore NJ dummy
# NCALLS   : Number of call-backs, Wave 1
# EMPFT    : Full-time employees, Wave 1
# EMPPT    : Part-time employees, Wave 1
# NMGRS    : Managers/assistant managers, Wave 1
# WAGE_ST  : Starting wage, Wave 1
# INCTIME  : Weeks to first raise, Wave 1
# FIRSTINC : Amount of first raise, Wave 1
# BONUS    : Sign-on bonus dummy, Wave 1
# PCTAFF   : % workers affected by minimum wage, Wave 1
# MEALS    : Free meal dummy, Wave 1
# OPEN     : Hour of opening, Wave 1
# HRSOPEN  : Hours open per day, Wave 1
# PSODA    : Price of medium soda, Wave 1
# PFRY     : Price of large fries, Wave 1
# PENTREE  : Price of entree, Wave 1
# NREGS    : Number of cash registers, Wave 1
# NREGS11  : Registers open at 11am, Wave 1
# TYPE2    : Type of Wave 2 interview
# STATUS2  : Status of Wave 2 interview (0=re-interview, 1=closed, 2=temp closed, etc.)
# DATE2    : Date of Wave 2 interview
# NCALLS2  : Number of call-backs, Wave 2
# EMPFT2   : Full-time employees, Wave 2
# EMPPT2   : Part-time employees, Wave 2
# NMGRS2   : Managers, Wave 2
# WAGE_ST2 : Starting wage, Wave 2
# INCTIME2 : Weeks to first raise, Wave 2
# FIRSTIN2 : Amount of first raise, Wave 2
# SPECIAL2 : Special promotion, Wave 2
# MEALS2   : Free meal dummy, Wave 2
# OPEN2R   : Hour of opening, Wave 2
# HRSOPEN2 : Hours open per day, Wave 2
# PSODA2   : Price of medium soda, Wave 2
# PFRY2    : Price of large fries, Wave 2
# PENTREE2 : Price of entree, Wave 2
# NREGS2   : Number of cash registers, Wave 2
# NREGS112 : Registers open at 11am, Wave 2

col_names = [
    'SHEET', 'CHAIN', 'CO_OWNED', 'STATE',
    'SOUTHJ', 'CENTRALJ', 'NORTHJ', 'PA1', 'PA2', 'SHORE',
    'NCALLS', 'EMPFT', 'EMPPT', 'NMGRS', 'WAGE_ST',
    'INCTIME', 'FIRSTINC', 'BONUS', 'PCTAFF', 'MEALS',
    'OPEN', 'HRSOPEN', 'PSODA', 'PFRY', 'PENTREE',
    'NREGS', 'NREGS11',
    'TYPE2', 'STATUS2', 'DATE2', 'NCALLS2',
    'EMPFT2', 'EMPPT2', 'NMGRS2', 'WAGE_ST2',
    'INCTIME2', 'FIRSTIN2', 'SPECIAL2', 'MEALS2',
    'OPEN2R', 'HRSOPEN2', 'PSODA2', 'PFRY2', 'PENTREE2',
    'NREGS2', 'NREGS112'
]

try:
    # njmin's public.dat is whitespace-delimited
    df_raw = pd.read_csv(
        'data/raw/public.dat',
        sep=r'\s+',
        header=None,
        names=col_names,
        na_values=['.', '999', '99.99']
    )
    print(f'Raw data loaded successfully! Shape: {df_raw.shape}')

    print('\nSTATUS2 column check:')
    print(df_raw['STATUS2'].value_counts(dropna=False))
    print(f'STATUS2 dtype: {df_raw['STATUS2'].dtype}')

except FileNotFoundError:
    print('public.dat not found. Checking available files...')
    for root, dirs_list, files in os.walk('data/raw/'):
        for file in files:
            print(f'  Found: {os.path.join(root, file)}')

print()
print('First 5 rows of raw data:')
df_raw.head()

Raw data loaded successfully! Shape: (410, 46)

STATUS2 column check:
STATUS2
1    399
3      6
2      2
4      1
0      1
5      1
Name: count, dtype: int64
STATUS2 dtype: int64

First 5 rows of raw data:


,SHEET,CHAIN,CO_OWNED,STATE,SOUTHJ,CENTRALJ,NORTHJ,PA1,PA2,SHORE,...,FIRSTIN2,SPECIAL2,MEALS2,OPEN2R,HRSOPEN2,PSODA2,PFRY2,PENTREE2,NREGS2,NREGS112
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1.0,2.0,6.5,16.5,1.03,NaN,0.94,4.0,4.0
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0.0,2.0,10.0,13.0,1.01,0.89,2.35,4.0,4.0
2,506,2,1,0,0,0,0,1,0,0,...,0.25,NaN,1.0,11.0,11.0,0.95,0.74,2.33,4.0,3.0
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,0.92,0.79,0.87,2.0,2.0
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,1.01,0.84,0.95,2.0,2.0


In [6]:
print('='*50)
print('Dataset Overview')
print('='*50)
print(f'Rows (stores)   : {df_raw.shape[0]}')
print(f'Columns (variables): {df_raw.shape[1]}')
print()
print('Store count by state:')
print(df_raw['STATE'].value_counts().rename({1: 'New Jersey (NJ)', 0: 'Pennsylvania (PA)'}))
print()
print('Store count by chain:')
chain_map = {1: 'Burger King', 2: 'KFC', 3: "Roy Rogers", 4: "Wendy's"}
print(df_raw['CHAIN'].map(chain_map).value_counts())

Dataset Overview
Rows (stores)   : 410
Columns (variables): 46

Store count by state:
STATE
New Jersey (NJ)      331
Pennsylvania (PA)     79
Name: count, dtype: int64

Store count by chain:
CHAIN
Burger King    171
Roy Rogers      99
KFC             80
Wendy's         60
Name: count, dtype: int64


In [7]:
# Check for missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('Variables with missing values:')
print(missing_df)

Variables with missing values:
          Missing Count  Missing %
EMPFT                 6       1.46
EMPPT                 4       0.98
NMGRS                 6       1.46
WAGE_ST              20       4.88
INCTIME              31       7.56
FIRSTINC             43      10.49
PCTAFF               44      10.73
PSODA                 8       1.95
PFRY                 17       4.15
PENTREE              12       2.93
NREGS                 6       1.46
NREGS11              12       2.93
NCALLS2             249      60.73
EMPFT2               12       2.93
EMPPT2               10       2.44
NMGRS2                6       1.46
WAGE_ST2             21       5.12
INCTIME2             66      16.10
FIRSTIN2             80      19.51
SPECIAL2             18       4.39
MEALS2               11       2.68
OPEN2R               11       2.68
HRSOPEN2             11       2.68
PSODA2               22       5.37
PFRY2                28       6.83
PENTREE2             24       5.85
NREGS2               22 

In [14]:
# ============================================================
# Construct core variables from Card & Krueger (1994)
# ============================================================

df = df_raw.copy()

# 1. Full-Time Equivalent (FTE) employment
#    FTE = full-time employees + 0.5 * part-time employees + managers
#    (Card & Krueger, 1994, Section II)
df['FTE1'] = df['EMPFT'] + 0.5 * df['EMPPT'] + df['NMGRS']
df['FTE2'] = df['EMPFT2'] + 0.5 * df['EMPPT2'] + df['NMGRS2']

# 2. Change in FTE employment (the DiD outcome variable)
df['DFTE'] = df['FTE2'] - df['FTE1']

# 3. Group labels
df['GROUP'] = df['STATE'].map({1: 'New Jersey (Treatment)', 0: 'Pennsylvania (Control)'})

print(f'Initial dataframe shape (after FTE calculation): {df.shape}')

# 4. Drop stores with abnormal Wave 2 status
#    STATUS2: 0=re-interview, 1=closed, 2=temp closed, 3=relocated, 5=incomplete
#    Based on value counts, STATUS2 == 1 appears to be the correct filter for re-interviewed stores.
df_filtered_status = df[df['STATUS2'] == 1].copy() # Changed from == 0 to == 1
print(f"Shape after filtering for STATUS2 == 1 (re-interviewed stores): {df_filtered_status.shape}")
print("Details of df_filtered_status before dropping NaNs:")
print(df_filtered_status[['SHEET', 'STATE', 'STATUS2', 'EMPFT', 'EMPPT', 'NMGRS', 'EMPFT2', 'EMPPT2', 'NMGRS2', 'FTE1', 'FTE2']].head())

# 5. Drop rows with missing FTE (outcome variable)
df_clean = df_filtered_status.dropna(subset=['FTE1', 'FTE2'])

print(f'Original sample size : {len(df)}')
print(f'Cleaned sample size  : {len(df_clean)}')
print(f'Dropped observations : {len(df) - len(df_clean)}')
print()
print('Stores by state after cleaning:')
print(df_clean['GROUP'].value_counts())

Initial dataframe shape (after FTE calculation): (410, 50)
Shape after filtering for STATUS2 == 1 (re-interviewed stores): (399, 50)
Details of df_filtered_status before dropping NaNs:
   SHEET  STATE  STATUS2  EMPFT  EMPPT  NMGRS  EMPFT2  EMPPT2  NMGRS2   FTE1  \
0     46      0        1   30.0   15.0    3.0     3.5    35.0     3.0  40.50   
1     49      0        1    6.5    6.5    4.0     0.0    15.0     4.0  13.75   
2    506      0        1    3.0    7.0    2.0     3.0     7.0     4.0   8.50   
3     56      0        1   20.0   20.0    4.0     0.0    36.0     2.0  34.00   
4     61      0        1    6.0   26.0    5.0    28.0     3.0     6.0  24.00   

   FTE2  
0  24.0  
1  11.5  
2  10.5  
3  20.0  
4  35.5  
Original sample size : 410
Cleaned sample size  : 378
Dropped observations : 32

Stores by state after cleaning:
GROUP
New Jersey (Treatment)    304
Pennsylvania (Control)     74
Name: count, dtype: int64


In [15]:
# Summary statistics by state (replicating Card & Krueger Table 2)
key_vars = ['FTE1', 'FTE2', 'DFTE', 'EMPFT', 'EMPPT', 'NMGRS',
            'WAGE_ST', 'WAGE_ST2', 'HRSOPEN', 'PSODA', 'PFRY']

# Check if df_clean is empty before proceeding
if df_clean.empty:
    print('Warning: The cleaned DataFrame (df_clean) is empty. No descriptive statistics can be computed.')
    # Create an empty DataFrame with the expected structure to avoid the ValueError
    desc = pd.DataFrame(columns=['Pennsylvania (PA)', 'New Jersey (NJ)'], index=key_vars)
else:
    desc = df_clean.groupby('STATE')[key_vars].mean().T
    desc.columns = ['Pennsylvania (PA)', 'New Jersey (NJ)']
    desc = desc.round(3)

print('Descriptive Statistics — Group Means by State:')
print(desc)

Descriptive Statistics — Group Means by State:
          Pennsylvania (PA)  New Jersey (NJ)
FTE1                 23.446           20.583
FTE2                 21.382           21.241
DFTE                 -2.064            0.658
EMPFT                10.203            7.752
EMPPT                19.378           18.842
NMGRS                 3.554            3.411
WAGE_ST               4.632            4.612
WAGE_ST2              4.617            5.081
HRSOPEN              14.534           14.396
PSODA                 0.974            1.062
PFRY                  0.842            0.941


In [16]:
# Save as Parquet (recommended — preserves dtypes) and CSV (fallback)
df_clean.to_parquet('data/processed/ck1994_clean.parquet', index=False)
df_clean.to_csv('data/processed/ck1994_clean.csv', index=False)

print('Cleaned data saved:')
print('   data/processed/ck1994_clean.parquet')
print('   data/processed/ck1994_clean.csv')
print(f'\nFinal dataset shape: {df_clean.shape}')
print()
df_clean[['SHEET','STATE','GROUP','FTE1','FTE2','DFTE','WAGE_ST','WAGE_ST2']].head(10)

Cleaned data saved:
   data/processed/ck1994_clean.parquet
   data/processed/ck1994_clean.csv

Final dataset shape: (378, 50)



,SHEET,STATE,GROUP,FTE1,FTE2,DFTE,WAGE_ST,WAGE_ST2
0,46,0,Pennsylvania (Control),40.50,24.0,-16.50,NaN,4.30
1,49,0,Pennsylvania (Control),13.75,11.5,-2.25,NaN,4.45
2,506,0,Pennsylvania (Control),8.50,10.5,2.00,NaN,5.00
3,56,0,Pennsylvania (Control),34.00,20.0,-14.00,5.00,5.25
4,61,0,Pennsylvania (Control),24.00,35.5,11.50,5.50,4.75
6,445,0,Pennsylvania (Control),70.50,29.0,-41.50,5.00,4.75
7,451,0,Pennsylvania (Control),23.50,36.5,13.00,5.00,5.00
8,455,0,Pennsylvania (Control),11.00,11.0,0.00,5.25,5.00
9,458,0,Pennsylvania (Control),9.00,8.5,-0.50,5.00,5.00
10,462,0,Pennsylvania (Control),15.50,17.5,2.00,5.00,4.75
